In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass

from datetime import datetime, time
from Database.TPData import TPData, TPDataDa
from OrderBook.OrderBook import OrderBookSnaps
from SynthSpread.spreadviewer_class import SpreadSingle
from Strategies.Sparse_momentum.ob_attributes import OB_attributes, TR_attributes
import pandas as pd
from Utilities.excel_loaders import conn_out_xload_mac
from Utilities.dfutils import dict_iloc
from Utilities.Storage import get_curr_storage_path
from Utilities.func_utils import load_arguments

In [2]:
_INSTRUMENT = 'dem2'

In [3]:
start_date='2024-07-01'
end_date='2025-05-10'

start_date = datetime.strptime(start_date, '%Y-%m-%d').date()
end_date = datetime.strptime(end_date, '%Y-%m-%d').date()

# Selecting all trades since 2024

In [4]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='{start_date}' and datetime<='{end_date}' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 3839778 rows from source database.


In [5]:
print(df)

                   datetime     nanotime                           tradeid
0       2024-07-01 08:05:02  105774489.0  Eurex T7/G3BM082024-20240701/1/5
1       2024-07-01 08:05:08    3636096.0  Eurex T7/DEBM082024-20240701/2/1
2       2024-07-01 08:05:08    6713923.0  Eurex T7/DEBM082024-20240701/3/1
3       2024-07-01 08:05:11  807488069.0  Eurex T7/DEBM082024-20240701/4/1
4       2024-07-01 08:06:23  284000000.0                          15262964
...                     ...          ...                               ...
3839773 2025-05-09 18:53:42  748000000.0                          17901285
3839774 2025-05-09 18:54:57  630000000.0                          17901289
3839775 2025-05-09 18:56:23  950000000.0                          17901294
3839776 2025-05-09 18:56:53  797000000.0                          17901299
3839777 2025-05-09 18:59:58  705000000.0                          17901306

[3839778 rows x 3 columns]


In [6]:
df['date'] = df['datetime'].dt.date

### Get trades for instrument

In [7]:
from Database.DB_reader import Database
db = Database('timescaledb')

pred_name_b = f'obook_b_price_{_INSTRUMENT}'
pred_name_a = f'obook_a_price_{_INSTRUMENT}'

get_pred_id_query = "SELECT pred_id FROM public.experimental_predictors WHERE pred_name = :pred_name"
pred_id_result = conn.execute_general_query(get_pred_id_query, {'pred_name': pred_name_b})
pred_id_b = pred_id_result.iloc[0]['pred_id'] if not pred_id_result.empty else None

pred_id_result = conn.execute_general_query(get_pred_id_query, {'pred_name': pred_name_a})
pred_id_a = pred_id_result.iloc[0]['pred_id'] if not pred_id_result.empty else None

# Query for 'obook_a_price_dey1'
df_a = db.execute_general_query(
    f"SELECT * FROM public.experimental_dataset_entries WHERE pred_id = '{pred_id_a}' AND datetime>='{start_date}' and datetime<='{end_date}'"
)

# Query for 'obook_b_price_dey1'
df_b = db.execute_general_query(
    f"SELECT * FROM public.experimental_dataset_entries WHERE pred_id = '{pred_id_b}' AND  datetime>='{start_date}' and datetime<='{end_date}'"
)

df_a = df_a.sort_values(by=['datetime', 'nanotime', 'tradeid'])
df_b = df_b.sort_values(by=['datetime', 'nanotime', 'tradeid'])

# Merge the dataframes on 'timestamp'
merged_df = pd.merge(df_b[['datetime', 'nanotime', 'tradeid', 'pred_value']],
                      df_a[['datetime', 'nanotime', 'tradeid', 'pred_value']], on=['datetime', 'nanotime', 'tradeid'], suffixes=('_b_price', '_a_price'))

# Rename columns for clarity
merged_df.rename(columns={'pred_value_b_price': 'b_price', 'pred_value_a_price': 'a_price'}, inplace=True)

print(merged_df)

Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
                   datetime     nanotime                           tradeid  \
0       2024-07-01 08:05:02  105774489.0  Eurex T7/G3BM082024-20240701/1/5   
1       2024-07-01 08:05:08    3636096.0  Eurex T7/DEBM082024-20240701/2/1   
2       2024-07-01 08:05:08    6713923.0  Eurex T7/DEBM082024-20240701/3/1   
3       2024-07-01 08:05:11  807488069.0  Eurex T7/DEBM082024-20240701/4/1   
4       2024-07-01 08:06:23  284000000.0                          15262964   
...                     ...          ...                               ...   
3607098 2025-05-09 18:53:42  748000000.0                          17901285   
3607099 2025-05-09 18:54:57  630000000.0            

### Transformations

In [8]:
merged_df['b_price'] = merged_df['b_price'].astype(float)
merged_df['a_price'] = merged_df['a_price'].astype(float)

In [9]:
VI_col_name = 'VI_mid_std_200'

In [10]:
merged_df['mid'] = 0.5 * (merged_df['b_price'] + merged_df['a_price'])
merged_df[VI_col_name] = merged_df['mid'].rolling(200).std()
merged_df = merged_df.reset_index(drop=True)

In [11]:
window = 400

defined_bins = [-float('inf'), -3, -1.5, -0.49, 0.49, 1.5, 3, float('inf')]
# Labels: 7 labels corresponding to the 7 bins. The central bin gets label 0.
defined_labels = [-3, -2, -1, 0, 1, 2, 3]
num_generated_bins = len(defined_labels) # Should be 7

# Lists to store the distributions (each element will be a Pandas Series of counts)
long_dists_list = []
short_dists_list = []

# Your loop (assuming 'merged_df' is correctly populated and 'window' is set)
i = 0
for row_data in merged_df.to_dict(orient='records'): # Use a different variable name to avoid confusion
    # --- Your calculations for r_long and r_short ---
    # These calculations will be the same in each iteration if merged_df isn't changing based on `row_data`
    # However, `VI` changes, so adjusted returns change.
    r_long = merged_df.loc[i:i+window, 'b_price'] - merged_df.loc[i, 'a_price']
    r_short =  merged_df.loc[i, 'b_price'] - merged_df.loc[i:i+window, 'a_price']
    
    VI = row_data[VI_col_name] # VI from the current row

    # Handle the special case for VI < 0.05
    if VI < 0.05:
        # Append a zero distribution for long, and presumably for short as well?
        # The Series should be indexed by your defined labels.
        zero_distribution = pd.Series(np.zeros(num_generated_bins), index=defined_labels)
        long_dists_list.append(zero_distribution.copy()) # Append a copy
        short_dists_list.append(zero_distribution.copy()) # Assuming same for short
        i += 1
        continue # Skip further processing for this row

    # Calculate adjusted r_long and r_short
    # Adding a small epsilon to VI to prevent division by zero if VI is exactly 0
    adjusted_r_long = r_long / (VI + 1e-9) # Use a very small epsilon
    adjusted_r_short = r_short / (VI + 1e-9)

    # --- Bin the adjusted values ---
    # pd.cut will return a Series of categorical data with your defined_labels
    # include_lowest=True ensures the smallest values are included in the first bin
    binned_long = pd.cut(adjusted_r_long, bins=defined_bins, labels=defined_labels, right=True, include_lowest=True)
    binned_short = pd.cut(adjusted_r_short, bins=defined_bins, labels=defined_labels, right=True, include_lowest=True)

    # --- Calculate the distribution of the bins (counts for each label) ---
    # .value_counts() gives counts for each unique label present.
    # .reindex(defined_labels, fill_value=0) ensures all defined labels are present in the output Series,
    # even if some bins had zero occurrences in this specific `adjusted_r_long/short`.
    long_bin_distribution = binned_long.value_counts().reindex(defined_labels, fill_value=0).sort_index()
    short_bin_distribution = binned_short.value_counts().reindex(defined_labels, fill_value=0).sort_index()

    long_dists_list.append(long_bin_distribution)
    short_dists_list.append(short_bin_distribution)
    i += 1


In [12]:
long_prob = [(srs_dist/srs_dist.sum()).values[4:].sum() for srs_dist in long_dists_list]
short_prob = [(srs_dist/srs_dist.sum()).values[4:].sum() for srs_dist in short_dists_list]

In [13]:
merged_df['class_long_prob'] = pd.Series(long_prob, index=merged_df.index)
merged_df['class_short_prob'] = pd.Series(short_prob, index=merged_df.index)

# Saving into TimescaleDB


In [14]:
merged_df.columns

Index(['datetime', 'nanotime', 'tradeid', 'b_price', 'a_price', 'mid',
       'VI_mid_std_200', 'class_long_prob', 'class_short_prob'],
      dtype='object')

In [15]:
columns_dict = {
    'pred_id': ['VI_mid_std_200', 'target_class_long_prob_400_VI_mid_std_200_'+_INSTRUMENT, 'target_class_short_prob_400_VI_mid_std_200_'+_INSTRUMENT],
    'pred_name': ['VI_mid_std_200','class_long_prob', 'class_short_prob'],
    'description': ['Volatility index calculated on mid prices sampled on base timestamps', 'Long class probability calculated from distribution of aggress returns adjusted with std',  'Short class probability calculated from distribution of aggress returns adjusted with std']
}

# Insert predictor template

In [16]:
from sqlalchemy import text

# Assuming Database is a custom class or connection handler for TimescaleDB
conn = Database('timescaledb')

for pred_id, pred_name, desc in zip(columns_dict['pred_id'], columns_dict['pred_name'], columns_dict['description']):
    pred_name = pred_id
    main_contract = _INSTRUMENT
    description = desc
    location = r"EnergyTrading\Python\prefect_data_orchestrator\datamart_inject\class_prob_predictors.ipynb"
    prod_strategy = None
    author = "MartinScasny"
    additional = ""

    # Define the SQL INSERT statement as a string with placeholders
    stmt = """
    INSERT INTO public.experimental_predictors
    (pred_name, main_contract, description, "location", prod_strategy, author, additional, created_at)
    VALUES (:pred_name, :main_contract, :description, :location, :prod_strategy, :author, :additional, CURRENT_TIMESTAMP)
    """

    # Execute the query with parameters passed as a dictionary
    conn.execute_general_query(stmt, {
        'pred_name': pred_name,
        'main_contract': main_contract,
        'description': description,
        'location': location,
        'prod_strategy': prod_strategy,
        'author': author,
        'additional': additional
    })

print("Data insertion completed successfully.")

Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Data insertion completed successfully.


In [17]:
# Adjust type for nanotime to match "df"
merged_df['nanotime'] = merged_df['nanotime'].astype(object)

In [19]:
conn = Database('timescaledb')
for pred_id, pred_name, desc in zip(columns_dict['pred_id'], columns_dict['pred_name'], columns_dict['description']):
    get_pred_id_query = "SELECT pred_id FROM public.experimental_predictors WHERE pred_name = :pred_name"
    pred_id_result = conn.execute_general_query(get_pred_id_query, {'pred_name': pred_id})
    component_pred_id = pred_id_result.iloc[0]['pred_id'] if not pred_id_result.empty else None

    if component_pred_id is None:
        print(f"Error: Could not retrieve pred_id for {pred_id}. Skipping data insertion.")

    print(f"Retrieved pred_id: {component_pred_id} for predictor {pred_id}")

    results=[]
    # Step 1: Convert wide df to long format using melt
    df_merged = pd.merge(df, merged_df, on=['datetime', 'tradeid'], how='inner')
    df_long = merged_df.melt(
        id_vars=['datetime', 'nanotime', 'tradeid'],  # Keep these columns fixed
        value_vars=[pred_name],
        var_name='pred_name',
        value_name='pred_value'
    )     
    # Step 2: Add pred_name and additional columns if required
    df_long['pred_id'] = component_pred_id

    # Step 3: Sort by datetime for proper forward filling
    df_long = df_long.sort_values(by=['datetime', 'nanotime','pred_id'], ascending=[True, True, True])

    # Step 4: Forward fill within each day separately
    df_long['date_only'] = df_long['datetime'].dt.date
    df_long['pred_value'] = df_long.groupby(['pred_id', 'date_only'])['pred_value'].ffill()

    # Optionally drop rows where pred_value is still NaN after forward fill
    #df_long = df_long.dropna(subset=['pred_value'])

    # Step 5: Drop helper columns
    df_long = df_long.drop(columns=['date_only'])

    # Locate rows where pred_value is missing and set additional message
    mask_missing = df_long['pred_value'].isna()


    # Optional step: enforce data types explicitly
    df_long['pred_value'] = df_long['pred_value'].astype(float)

    df_long = df_long[['datetime', 'nanotime', 'tradeid', 'pred_id','pred_value']].reset_index(drop=True)


    # 2. Connect to TimescaleDB
    batch_size=100_000
    conn = Database('timescaledb')
    conn._connect()

    # 3. Insert in batches
    total_rows = len(df_long)
    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch = df_long.iloc[start:end]

        batch.to_sql('experimental_dataset_entries', conn.engine,schema='public', index=False, if_exists='append',method='multi')
        print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")

        print("🎉 All batches inserted successfully.")

Connected to the database timescaledb
Disconnected from the database timescaledb
Retrieved pred_id: 292 for predictor VI_mid_std_200


Connected to the database timescaledb
✅ Inserted rows 0 to 100000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 100000 to 200000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 200000 to 300000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 300000 to 400000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 400000 to 500000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 500000 to 600000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 600000 to 700000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 700000 to 800000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 800000 to 900000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 900000 to 1000000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 1000000 to 1100000 into TimescaleDB.
🎉 All batches inserted successful

In [ ]:
df_merged[df_merged['class_long_prob'] > 0]